In [ ]:
import os
import sounddevice as sd
import numpy as np
from faster_whisper import WhisperModel
import ollama
from IPython.display import clear_output
import time
from pydantic import BaseModel
import time


In [ ]:
# Test Audio Device is detected
print(sd.query_devices())

In [ ]:
# Initial Whisper Model Config
model_size = "small"
model = WhisperModel(model_size, device="cuda")

In [ ]:
#Fixed length recording audio
duration = 4.5  # seconds
fs = 16000
myrecording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait()
print("stop")


In [ ]:
temp = [1,2,3,4,5,6,7]

In [ ]:
temp[:5]

In [ ]:
temp[-2:]

In [ ]:
chunks = 1 2 3 4 5 6 7 8 9
window size is 5
and overlap size is 2

when we gewt he window, we get
chunks[:5]
aka
1 2 3 4 5
and this audio, will have an overlap with the next set we want to do
teh next set we want to process is going to be
4 5 6 7 8

so we want to remove
1 2 3
so to that that we set chunks to be
chunks[3:]

In [ ]:
temp

In [ ]:
temp[3:]

In [ ]:
self._chunks = self._chunks[-(self.window_size - self.overlap_size):]


In [ ]:
chunks[-num_we_want_to_maintain:]

In [ ]:
sd.play(myrecording, fs)

In [ ]:
segments, info = model.transcribe(myrecording.flatten(), beam_size = 5, word_timestamps= True)

seg_list = []
for segment in segments:
    seg_list.append(segment)
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

In [155]:
class AudioBuffer:
    def __init__(self, window_size = 5, overlap_size = 2, sample_rate = 16000):
        self.window_size = window_size
        self.overlap_size = overlap_size
        self.sample_rate = sample_rate
        self._chunks = []

    def append_chunk(self, audio_data):
        #chunk preprocessing goes here, before append
        self._chunks.append(audio_data.copy())

    def get_buffer_size(self):
        return len(self._chunks)
        
    def get_window(self):
        # return concatenated window if ready, else None
        if len(self._chunks) >= self.window_size:
            window = np.concatenate(self._chunks[:self.window_size])
            self._chunks = self._chunks[(self.window_size - self.overlap_size):]
            return window
        return None

In [156]:
class TranscriptionPipeline:
    def __init__(self, whisper_model, merge_selector):
        self.whisper = whisper_model
        self.merge_selector = merge_selector
        self.transcript_history = ""

    def process_audio(self, audio_window):
        segments, _ = self.whisper.transcribe(
            audio_window.flatten(),
            beam_size = 5)
        return " ".join(segment.text for segment in segments)

    def merge_with_history(self, new_transcript):
        #delegates merging
        self.transcript_history = self.merge_selector.merge(
            self.transcript_history, 
            new_transcript
        )
        return self.transcript_history

    def process_and_merge(self, audio_window):
        transcript = self.process_audio(audio_window)
        self.merge_with_history(transcript)

        return self.transcript_history
        
    

In [157]:
example_string = "This is an example...string that I am writing's about now. yep"
example_string2 = "This is an example... string two that I am writing's about now. yep"

import string
import re
example_string.translate(str.maketrans('', '', string.punctuation)).lower()

out1 = re.sub(r'\.{3}(?=\S|$)', '... ', example_string)
out2 = re.sub(r'\.{3}(?=\S|$)', '... ', example_string2)

print(out1)
print(out2)

This is an example... string that I am writing's about now. yep
This is an example... string two that I am writing's about now. yep


In [158]:
class MergeStrategySelector:
    def __init__(self, context_words = 10):
        self.context_words = context_words
        self.stats = {'exact': 0, 'fuzzy': 0, 'llm': 0}
        self.simple_strategy = SimpleMergeStrategy()
        self.llm_strategy = LLMMergeStrategy(model_name = 'llama3.2:3b', context_words = self.context_words)

    def merge(self, existing_text, new_text):
        
        new_text_clean = re.sub(r'\.{3}(?=\S|$)', '... ', new_text) #this should be somewhere else, tbd
        
        if not existing_text:
            return new_text_clean

        context_text = " ".join(existing_text.split()[-self.context_words:])

        
        
        match_info = self._analyse_overlap(context_text, new_text_clean, max_match_len = 10)
        
        if match_info['type'] == "exact":
            merged_output = self.simple_strategy.merge(existing_text, new_text_clean, match_info)
            self.stats["exact"] += 1
        elif match_info['type'] == 'llm':
            merged_output = self.llm_strategy.merge(existing_text, new_text_clean, match_info)
            self.stats["llm"] += 1

        return merged_output

    def _analyse_overlap(self, context_text, new_text, max_match_len = 10):

        context_text_analysis = context_text.translate(str.maketrans('', '', string.punctuation)).lower()
        new_text_analysis = new_text.translate(str.maketrans('', '', string.punctuation)).lower()
        
        context_words = context_text_analysis.split()
        new_words = new_text_analysis.split()
        
        match_len_limit = min(len(context_words), len(new_words), max_match_len)
        
        match_index = 0
        for i in range(match_len_limit, 0,- 1):
            v1 = context_words[-i:]
            v2 = new_words[:i]
            if v1 == v2:
                match_index = i
                break
        merge_type = "exact"
        if match_index == 0:
            merge_type = "llm"
            print("DEBUG: Exact Merging Failed")
            print(context_words[-match_len_limit:])
            print(new_words[:match_len_limit])
            
        
        return {'type': merge_type, 'overlap_length': match_index}

In [159]:
class SimpleMergeStrategy:
    def merge(self, existing_text, new_text, match_info):
        merged_list = existing_text.split() + new_text.split()[match_info['overlap_length']:]
        return " ".join(merged_list)

In [160]:
class merged_response(BaseModel):
  merged_output: str 


class LLMMergeStrategy:
    def __init__(self, model_name, context_words = 10):
        self.model_name = model_name
        self.context_words = context_words

        
    def merge(self, existing_text, new_text, match_info):

        if not existing_text:
            return new_text
        
        context_fragment = " ".join(existing_text.split()[-self.context_words:])
        
        llm_message = '''Here are the two transcript fragments
        1: ''' + context_fragment + '''
        2: ''' + new_text


        
        response = ollama.chat(model=self.model_name, format = merged_response.model_json_schema(), messages=[
                {'role': 'system', 'content': '''You merge overlapping transcript fragments. The end of transcript 1 overlaps with the start of transcript 2. Remove the duplication and return ONLY the merged text.
                
                Rules:
                - Output ONLY the merged text
                - No explanations, notes, or commentary
                - If there is no clear overlap, return transcript 2 unchanged
                
                Example:
                1: The cat sat on the mat
                2: on the mat and then fell asleep
                Output: The cat sat on the mat and then fell asleep'''}
                ,
                {'role': 'user', 'content': llm_message}
            ])
    
        output_text = existing_text[:-len(context_fragment)] + " " + merged_response.model_validate_json(response.message.content).merged_output
    
        return output_text

In [ ]:
test = None

In [ ]:
if not test:
    print("yup")

In [ ]:
ollama.pull('llama3.2:3b')

In [ ]:
print(ollama.list())

In [ ]:
merge_selector.stats

In [167]:
type(window)

numpy.ndarray

In [162]:
whisper_model = WhisperModel(model_size, device="cuda")
audio_buffer = AudioBuffer(window_size = 5, overlap_size = 2)
merge_selector = MergeStrategySelector(context_words = 10)
pipeline = TranscriptionPipeline(whisper_model, merge_selector)

def audio_callback(indata, frames, times, status):
    audio_buffer.append_chunk(indata)

with sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000):
    while True:
        sd.sleep(100)
        window = audio_buffer.get_window()
        if window is None:
            continue
        transcribed_text = pipeline.process_and_merge(window)
        print(audio_buffer.get_buffer_size())
        print(transcribed_text)

2
 Again, this time my chair's bit squeaky, and also, um... 
DEBUG: Exact Merging Failed
['again', 'this', 'time', 'my', 'chairs', 'bit', 'squeaky', 'and', 'also', 'um']
['and', 'also', 'i', 'now', 'have', 'when', 'there', 'is', 'a', 'successful']
4
 A Again, this time my chair's bit squeaky, and also, I now have, when there is a successful
2
A Again, this time my chair's bit squeaky, and also, I now have, when there is a successful buffer or when the buffer is
DEBUG: Exact Merging Failed
['there', 'is', 'a', 'successful', 'buffer', 'or', 'when', 'the', 'buffer', 'is']
['buffer', 'when', 'the', 'buffer', 'is', 'basically', 'when', 'were', 'actually', 'transcripting']
4
A Again, this time my chair's bit squeaky, and also, I now have, when  there is a successful buffer or when the buffer is basically when we're actually transcripting something
2
A Again, this time my chair's bit squeaky, and also, I now have, when there is a successful buffer or when the buffer is basically when we're ac

KeyboardInterrupt: 

In [163]:
merge_selector.stats

{'exact': 34, 'fuzzy': 0, 'llm': 25}

In [ ]:
class merged_response(BaseModel):
  merged_output: str 

def text_output():
    #clear_output()

    global text_string
    global transcript_iter
    
    print("Transcript_iter - " + str(transcript_iter)) 
    print("----")
    print("text_string initially is: \n" + text_string)
    
    last_10_words = " ".join(text_string.split()[-10:])
    text_string = text_string[:-len(last_10_words)]

    print("----")
    print("last_10_words is: \n" + last_10_words)
    print("----")
    print("text_string with words removed: \n" + text_string)
    print("----")
    
    
    section_a = last_10_words
    section_b = whisper_out

    print("whisper_out is: \n" + whisper_out)
    print("----")
    llm_message = '''Here are the two transcript fragments
    1: ''' + section_a + '''
    2: ''' + section_b

    start = time.time()
    response = ollama.chat(model='llama3.2:3b', format = merged_response.model_json_schema(), messages=[
        {'role': 'system', 'content': '''You merge overlapping transcript fragments. The end of transcript 1 overlaps with the start of transcript 2. Remove the duplication and return ONLY the merged text.
        
        Rules:
        - Output ONLY the merged text
        - No explanations, notes, or commentary
        - If there is no clear overlap, return transcript 2 unchanged
        
        Example:
        1: The cat sat on the mat
        2: on the mat and then fell asleep
        Output: The cat sat on the mat and then fell asleep'''}
        ,
        {'role': 'user', 'content': llm_message}
    ])

    llm_time = time.time() - start
    print(f"LLM took: {llm_time:.3f}s")

    text_string += " " + merged_response.model_validate_json(response.message.content).merged_output
    transcript_iter += 1

    print("Updated text_string is: \n" + text_string)
    
    


In [ ]:
test_audio = 
segments, info = model.transcribe(full_audio.flatten(), beam_size = 5)
for segment in segments:
            whisper_out += segment.text

In [ ]:
text_string = ""
transcript_iter = 1

audio_chunks = []
def audio_callback(indata, frames, time, status):
    audio_chunks.append(indata.copy())

In [ ]:
stream = sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000)

In [ ]:
stream.start()
while True:
    if len(audio_chunks) >= 5:
        full_audio = np.concatenate(audio_chunks)
        del audio_chunks[0:3]
        start = time.time()
        segments, info = model.transcribe(full_audio.flatten(), beam_size = 5)
        whisper_out = ""
        for segment in segments:
            whisper_out += segment.text
        whisper_time = time.time() - start
        print(f"Whisper took: {whisper_time:.3f}s")
        text_output()
    sd.sleep(100)

In [ ]:
stream.stop()

In [ ]:
#playing with vad

import torch

torch.set_num_threads(1)

In [ ]:

model_vad, utils = torch.hub.load(
    repo_or_dir= 'snakers4/silero-vad',
    model = 'silero_vad')
    

In [ ]:
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

In [ ]:
get_speech_timestamps

In [ ]:
audio_tensor = torch.from_numpy(myrecording.flatten()).float()

In [ ]:
sd.play(myrecording, fs)

In [ ]:
speech_timestamps = get_speech_timestamps(
  audio = audio_tensor,
  model = model_vad,
  return_seconds=True,  # Return speech timestamps in seconds (default is samples)
  min_speech_duration_ms=5,  # Minimum speech chunk duration)
  min_silence_duration_ms = 5
)
    

speech_timestamps